# Python и машинное обучение: нейронные сети и компьютерное зрение

## Модуль 4. Поиск похожих изображений

Загрузим датасет CalTech101.

In [ ]:
!pip install gdown

In [ ]:
!mkdir ./datasets
!gdown https://drive.google.com/uc?id=1PH41pa0MNI2iGTURGwPHVoshCcerrPMk --output ./datasets/caltech101.zip
!unzip ./datasets/caltech101.zip -d ./datasets
!rm -rf "./datasets/caltech-101/BACKGROUND_Google"
!echo 'All done!'

In [ ]:
!ls -al ./datasets/caltech-101

In [ ]:
import os, shutil

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets,transforms
from torch.nn.functional import normalize

from torchinfo import summary
from torchmetrics import Accuracy, AUROC

from torch.utils.data.sampler import SubsetRandomSampler
import torch.nn.functional as F

from PIL import Image

import torchvision.models as models

import requests
imagenet_classes = requests.get('https://files.fast.ai/models/imagenet_class_index.json').json()

from numpy.linalg import norm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

device = "cuda" if torch.cuda.is_available() else \
    "mps" if torch.backends.mps.is_built() else "cpu"
device

In [ ]:
!pip install torchinfo torchmetrics

In [ ]:
IMAGE_SIZE = (224,224)
BASE_DIR = './datasets/caltech-101'

data_transforms = transforms.Compose([
    transforms.Resize(size=IMAGE_SIZE), # делаем все картинки квадратными
    transforms.ToTensor(), # преобразуем в тензор
#     transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

class ImageFolderWithPaths(datasets.ImageFolder):
    def __getitem__(self, index):
        img, label = super().__getitem__(index)
        path = self.imgs[index][0]
        return (img, label ,path)

img_data = ImageFolderWithPaths(root=BASE_DIR,
                                  transform=data_transforms, 
                                  target_transform=None)

print(f"Total number of images: {len(img_data)}")

## 1. Поиск дубликатов изображений

Для поиска дубликатов изображений можно использовать различные методы, такие как сравнение гистограмм, использование алгоритмов хэширования изображений (например, Perceptual Hashing), или же более сложные методы на основе нейронных сетей, такие как использование предобученных моделей для извлечения признаков и сравнения этих признаков.   

In [ ]:
try:
    import imagehash
except ImportError:
    %pip install -q ImageHash
    import imagehash

import hashlib

def file_md5(path: str, chunk_size: int = 8192) -> str:
    md5 = hashlib.md5()
    with open(path, "rb") as file:
        while True:
            chunk = file.read(chunk_size)
            if not chunk:
                break
            md5.update(chunk)
    return md5.hexdigest()


def build_hash_dataframe(image_dataset) -> pd.DataFrame:
    rows = []
    for path, class_id in image_dataset.imgs:
        with Image.open(path) as image:
            image = image.convert("RGB")
            rows.append({
                "path": path,
                "class_id": class_id,
                "class_name": os.path.basename(os.path.dirname(path)),
                "md5": file_md5(path),
                "phash": str(imagehash.phash(image, hash_size=16)),
                "whash": str(imagehash.whash(image, hash_size=16, mode="haar")),
            })
    return pd.DataFrame(rows).set_index("path")


hash_df = build_hash_dataframe(img_data)
hash_df

In [ ]:
np.random.seed(42)
sample_paths = np.random.choice(hash_df.index.to_numpy(), size=2, replace=False)

images = {}

for query_path in sample_paths:
    print(f"\nЗапрос: {os.path.basename(query_path)}")
    print(f"Класс: {hash_df.loc[query_path, 'class_name']}")
    
    images[query_path] = {'md5': file_md5(query_path),
                         'phash': str(imagehash.phash(Image.open(query_path), hash_size=16)),
                         'whash': str(imagehash.whash(Image.open(query_path), hash_size=16, mode="haar"))}
    print("Хеши: md5: {},\n phash: {},\n whash: {}".format(
        images[query_path]["md5"],
        images[query_path]["phash"],
        images[query_path]["whash"],
    ))

    plt.imshow(Image.open(query_path))
    plt.axis("off")
    plt.show()


In [ ]:
from sklearn.neighbors import NearestNeighbors

HASH_COLUMNS = ["md5", "phash", "whash"]

kNNs = {}

def hash_to_bit_features(series: pd.Series) -> np.ndarray:
    return np.array(
        [np.unpackbits(np.frombuffer(bytes.fromhex(hash_value), dtype=np.uint8)) for hash_value in series],
        dtype=np.uint8,
    )

for hash_name in HASH_COLUMNS:
    features = hash_to_bit_features(hash_df[hash_name])
    neighbors = NearestNeighbors(metric="hamming", algorithm="brute")
    neighbors.fit(features)
    kNNs[hash_name] = neighbors

kNNs

In [ ]:
n_neighbors = 5

for image in images:
    print(f"\nЗапрос: {os.path.basename(image)}")
    for hash_name in HASH_COLUMNS:
        print(f"\nМетрика: {hash_name}")
        fig, axs = plt.subplots(1, n_neighbors, figsize=(15, 5))
        query_features = hash_to_bit_features(pd.Series([images[image][hash_name]]))
        distances, indices = kNNs[hash_name].kneighbors(query_features, n_neighbors=n_neighbors)
        for distance, (dist, index) in enumerate(zip(distances[0], indices[0])):
            similar_image_path = hash_df.index[index]
            axs[distance].set_title(f"Расстояние: {distance:.4f}")
            axs[distance].imshow(Image.open(similar_image_path))
            axs[distance].axis("off")
        plt.show()

**ПРАКТИКА**

Возьмите изображения 'airplane1.jpg', 'airplane2.jpg', 'airplane3.jpg' и 'leopards.jpg' и найдите их дубликаты и/или похожие изображения в датасете, используя алгоритмы хэширования. Сравните результаты и обсудите, какие методы работают лучше для данного случая.

In [ ]:
# ваш код здесь



## 2. Поиск похожих изображений с помощью сверточных нейронных сетей

Для поиска похожих изображений можно использовать методы, аналогичные тем, что используются для поиска дубликатов, но с более гибкими критериями сравнения. Например, можно использовать косинусное сходство между векторами признаков, извлеченными из изображений с помощью предобученных моделей (например, ResNet, VGG). Также можно использовать алгоритмы кластеризации для группировки похожих изображений вместе.

In [ ]:
model_full = models.resnet50(weights='DEFAULT').to(device)
print(model_full)

In [ ]:
modules=list(model_full.children())[:-1]
model_no_fc=nn.Sequential(*modules)
for p in model_no_fc.parameters():
    p.requires_grad = False
    
print(model_no_fc)

In [ ]:
summary(model_no_fc,
        input_size=(1, 3, 224, 224),
        col_names=["input_size", "output_size", "num_params"],
        device=device
       )

In [ ]:
np.random.seed(20231221)
ix_random_image = np.random.choice(len(img_data))

img, label, path = img_data[ix_random_image]
print(f"Image filename: {img_data.imgs[ix_random_image]}")
display(transforms.ToPILImage()(img))

In [ ]:
model_full.eval()
results = model_full(img.unsqueeze(0).to(device))

top = torch.sort(F.softmax(results, dim=1)[0] * 100, descending=True)
predictions = [f"{imagenet_classes[str(ix.cpu().item())][1]} - {pct:.2f}%" \
               for pct, ix in zip(*top) ][:5]
predictions

А теперь получим эмбеддинг:

In [ ]:
fc_input = model_no_fc(img.unsqueeze(0).to(device))
print(fc_input.shape, f"Max: {fc_input.max()}, min: {fc_input.min()}")

fc_input = torch.flatten( fc_input, start_dim=1 )[0]

embedding = fc_input / torch.sqrt(fc_input.dot(fc_input)) # нормализуем
print(embedding)


**ПРАКТИКА**

Напишите функцию, которая будет принимать на вход минибатч из изображений и возвращать pandas dataframe, содержащий имя файла в качестве индекса и 2048 признаков из ембеддинга. Названия фичей должны начинаться с префикса ```f...```, например, ```f0, f1, ..., f2048```.

In [ ]:
def get_embeddings(imgs: torch.Tensor, paths) -> pd.DataFrame:
    with torch.no_grad():
        fc_input = model_no_fc(imgs.to(device))
        fc_input = torch.flatten(fc_input, start_dim=1)
        fc_input = fc_input.cpu().numpy()

    # правильная L2-нормализация по строкам
    norms = np.sqrt(np.sum(fc_input ** 2, axis=1, keepdims=True))
    embeddings = fc_input / (norms + 1e-12)

    df_embeddings = pd.DataFrame(
        embeddings,
        index=paths
    )
    return df_embeddings
    
    

In [ ]:
BATCH_SIZE = 20
loader = DataLoader(dataset=img_data, batch_size=BATCH_SIZE, shuffle=True)

imgs, _, paths = next(iter(loader))

get_embeddings(imgs.to(device), paths)

In [ ]:
%%time
get_embeddings(imgs, paths)

In [ ]:
df = None
for imgs, _, paths in loader:
    df_embds = get_embeddings(imgs, paths)
    if df is None:
        df = df_embds
    else:
        df = pd.concat([df, df_embds])
        
df

In [ ]:
from sklearn.neighbors import NearestNeighbors

neighbors = NearestNeighbors(n_neighbors=10,
                             algorithm='brute',
                             metric='euclidean').fit(df)

In [ ]:
np.random.seed(2023122102)
ix_random_image = np.random.choice(len(img_data))

img = Image.open(df.iloc[ ix_random_image ].name)
display(img)

In [ ]:
distances, indices = neighbors.kneighbors(df.iloc[ [ix_random_image] ])
print(distances)
print(indices)
df.iloc[ indices[0] ]

In [ ]:
fig = plt.figure(figsize=(20, 2))
for idx, (filename, row) in enumerate(df.iloc[ indices[0] ].iterrows()):
    ax = fig.add_subplot(1, 10, idx+1, xticks=[], yticks=[])
    ax.imshow(Image.open(row.name))
    class_ = os.path.split(os.path.split(filename)[0])[1]
    ax.set_title(class_)
    
plt.show()

**ПРАКТИКА**

Загрузите в директорию с тетрадью любое изображение из интернета или с жесткого диска. Попробуйте также на изображениях 'airplane1.jpg', 'airplane2.jpg', 'airplane3.jpg' и 'leopards.jpg'. Найдите похожие изображения в датасете CalTech101 и выведите их на экран. Сделайте выводы.

In [ ]:
# ваш код здесь



## 3. Использование CLIP для поиска похожих изображений

Нейросеть CLIP от OpenAI формирует векторные представления изображений и текстов и позволяет сопоставлять их. Это может быть использовано для поиска похожих изображений на основе их семантического содержания.

In [ ]:
!pip install transformers

In [ ]:
all_paths = df.index.to_numpy()

np.random.seed(20260423)
query_path = np.random.choice(all_paths)

image = Image.open(query_path).convert("RGB")
display(image)

In [ ]:
# Название класса берем из имени родительской папки
class_names = sorted({os.path.basename(os.path.dirname(p)) for p in all_paths})

# Кандидаты текстовых описаний
candidate_texts = [f"a photo of a {cls.replace('_', ' ')}" for cls in class_names]
candidate_texts[:10]

In [ ]:
from transformers import CLIPModel, CLIPProcessor

# загружаем модель и процессор
clip_model_name = "openai/clip-vit-base-patch32"

clip_model = CLIPModel.from_pretrained(clip_model_name).to(device)
clip_processor = CLIPProcessor.from_pretrained(clip_model_name)

In [ ]:
# Подготовка входов для CLIP
inputs = clip_processor(
    text=candidate_texts,
    images=image,
    return_tensors="pt",
    padding=True
)

inputs = {k: v.to(device) for k, v in inputs.items()}

# Считаем похожесть image <-> text
with torch.no_grad():
    outputs = clip_model(**inputs)
    logits_per_image = outputs.logits_per_image  # shape: [1, num_texts]
    probs = logits_per_image.softmax(dim=1).cpu().numpy()[0]

# Топ-5 текстовых описаний
top_k = 5
top_idx = np.argsort(probs)[::-1][:top_k]

print("Файл:", query_path)
print("\nТоп текстовых описаний от CLIP:")
for rank, i in enumerate(top_idx, start=1):
    print(f"{rank}. {candidate_texts[i]}   |   p = {probs[i]:.4f}")

# Визуализация
plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.axis("off")
plt.title(
    "CLIP prediction:\n"
    f"{candidate_texts[top_idx[0]]}\n"
    f"(p = {probs[top_idx[0]]:.4f})"
)
plt.show()

In [ ]:
def get_clip_embeddings(imgs: torch.Tensor, paths) -> pd.DataFrame:
    """
    imgs: батч тензоров после вашего DataLoader
    paths: пути к изображениям
    """

    pil_images = [Image.open(path).convert("RGB") for path in paths]

    inputs = clip_processor(
        images=pil_images,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        image_features = clip_model.get_image_features(**inputs)

        # В некоторых версиях transformers здесь может вернуться
        # не тензор, а объект с уже готовым pooler_output/image_embeds.
        if hasattr(image_features, "image_embeds"):
            image_features = image_features.image_embeds
        elif hasattr(image_features, "pooler_output"):
            pooler_output = image_features.pooler_output
            if pooler_output.shape[-1] == clip_model.visual_projection.in_features:
                image_features = clip_model.visual_projection(pooler_output)
            else:
                image_features = pooler_output

        image_features = image_features.detach().cpu().numpy()

    norms = np.sqrt(np.sum(image_features ** 2, axis=1, keepdims=True))
    embeddings = image_features / (norms + 1e-12)

    df_embeddings = pd.DataFrame(
        embeddings,
        index=paths
    )
    return df_embeddings

df_clip = None

for imgs, _, paths in loader:
    df_embds = get_clip_embeddings(imgs, paths)
    if df_clip is None:
        df_clip = df_embds
    else:
        df_clip = pd.concat([df_clip, df_embds])

df_clip

In [ ]:
neighbors_clip = NearestNeighbors(
    n_neighbors=10,
    algorithm="brute",
    metric="cosine"
).fit(df_clip)

In [ ]:
N_NEIGHBORS_TO_SHOW = 9

np.random.seed(2026042301)
ix_random_image = np.random.choice(len(df_clip))

query_path = df_clip.iloc[ix_random_image].name

dist_clip_all, ind_clip_all = neighbors_clip.kneighbors(
    df_clip.iloc[[ix_random_image]],
    n_neighbors=N_NEIGHBORS_TO_SHOW + 1
)

# убираем саму query-картинку
dist_clip = dist_clip_all[0][1:]
ind_clip = ind_clip_all[0][1:]

print("Запрос:", query_path)
print("\nCosine distances (CLIP):")
print(np.round(dist_clip, 4))

fig = plt.figure(figsize=(18, 4))

# QUERY
ax = fig.add_subplot(2, N_NEIGHBORS_TO_SHOW, 1, xticks=[], yticks=[])
ax.imshow(Image.open(query_path))
ax.set_title("QUERY")

for j in range(2, N_NEIGHBORS_TO_SHOW + 1):
    ax = fig.add_subplot(2, N_NEIGHBORS_TO_SHOW, j, xticks=[], yticks=[])
    ax.axis("off")

# CLIP neighbors
for idx, row_idx in enumerate(ind_clip):
    filename = df_clip.iloc[row_idx].name
    class_ = os.path.split(os.path.split(filename)[0])[1]

    ax = fig.add_subplot(2, N_NEIGHBORS_TO_SHOW, N_NEIGHBORS_TO_SHOW + idx + 1, xticks=[], yticks=[])
    ax.imshow(Image.open(filename))
    ax.set_title(f"CLIP #{idx+1}\n{class_}")

plt.tight_layout()
plt.show()

**ПРАКТИКА**

Выполните сравнительный поиск похожих изображений в CalTech101 с помощью сверточных сетей и CLIP на любых понравившихся вам изображениях: загруженных ранее и/или 'airplane1.jpg', 'airplane2.jpg', 'airplane3.jpg' и 'leopards.jpg'. Сделайте выводы.

In [ ]:
# ваш код здесь




## 4. Поиск объектов с похожими признаками

Для поиска объектов с похожими признаками можно использовать сверточные нейронные сети для извлечения признаков из изображений и затем сравнивать эти признаки между собой. Например, можно использовать предобученные модели (например, ResNet, VGG) для получения векторов признаков и затем использовать косинусное сходство или евклидово расстояние для сравнения этих векторов. Также можно использовать алгоритмы кластеризации для группировки изображений с похожими признаками вместе.

In [ ]:
face_like_paths = [
    p for p in df.index
    if os.path.basename(os.path.dirname(p)) in {"Faces", "Faces_easy"}
]
print("Всего лиц:", len(face_like_paths))
face_like_paths


In [ ]:
# визуализируем 20 случайных картинок из этих папок
np.random.seed(20260423)
fig, axs = plt.subplots(4, 5, figsize=(15, 12))
for ax, path in zip(axs.ravel(), np.random.choice(face_like_paths, 20, replace=False)):
    ax.set_title(os.path.basename(os.path.dirname(path)) + '/' + os.path.basename(path))
    ax.imshow(plt.imread(path))
    ax.axis('off')
plt.show()

### 4.1 Извлечение признаков при помощи специализированной модели

Посмотрим на примере поиска самых улыбающихся лиц. Для этого нам нужно сначала оценить "улыбчивость" каждого лица. Воспользуемся для этого предобученной моделью, которая выдаёт smile_score от 0 до 1.

In [ ]:
!pip install mediapipe requests

In [ ]:
%pip install -q mediapipe requests

In [ ]:
import os
import requests

MODEL_PATH = "face_landmarker.task"
MODEL_URL = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task"

if not os.path.exists(MODEL_PATH):
    r = requests.get(MODEL_URL, timeout=60)
    r.raise_for_status()
    with open(MODEL_PATH, "wb") as f:
        f.write(r.content)

print("Model ready:", MODEL_PATH)

In [ ]:
import mediapipe as mp
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=VisionRunningMode.IMAGE,
    num_faces=1,
    output_face_blendshapes=True,
    output_facial_transformation_matrixes=False
)

face_landmarker = FaceLandmarker.create_from_options(options)

In [ ]:
# ищем нужный путь внутри df.index, как и раньше в вашей тетради
img_path = [p for p in df.index if "Faces_easy/image_0306.jpg" in p][0]
# img_path = [p for p in df.index if "Faces/image_0372.jpg" in p][0]


img = np.array(Image.open(img_path).convert("RGB"))
h, w = img.shape[:2]

mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img)
result = face_landmarker.detect(mp_image)

plt.figure(figsize=(8, 8))
plt.imshow(img)

if result.face_landmarks:
    face_lms = result.face_landmarks[0]

    xs = [lm.x * w for lm in face_lms]
    ys = [lm.y * h for lm in face_lms]

    plt.scatter(xs, ys, s=4)

    print(f"Landmarks detected: {len(face_lms)}")

    if result.face_blendshapes:
        scores = {c.category_name: c.score for c in result.face_blendshapes[0]}
        smile_left = scores.get("mouthSmileLeft", 0.0)
        smile_right = scores.get("mouthSmileRight", 0.0)
        smile_score = (smile_left + smile_right) / 2.0

        print(f"mouthSmileLeft : {smile_left:.3f}")
        print(f"mouthSmileRight: {smile_right:.3f}")
        print(f"smile_score    : {smile_score:.3f}")
else:
    print("Лицо не найдено")

plt.title("MediaPipe Face Landmarker: dense face landmarks")
plt.axis("off")
plt.show()

In [ ]:
# Кандидаты: только "лицевые" классы Caltech101
face_like_paths = [
    p for p in df.index
    if os.path.basename(os.path.dirname(p)) in {"Faces", "Faces_easy"}
]

def get_smile_score(path):
    """
    Возвращает словарь с smile_score для одного изображения.
    Если лицо не найдено, возвращает None.
    """
    try:
        img = np.array(Image.open(path).convert("RGB"))
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img)
        result = face_landmarker.detect(mp_image)

        if not result.face_landmarks:
            return None

        blend = {}
        if result.face_blendshapes:
            blend = {c.category_name: c.score for c in result.face_blendshapes[0]}

        smile_left = blend.get("mouthSmileLeft", 0.0)
        smile_right = blend.get("mouthSmileRight", 0.0)
        smile_score = (smile_left + smile_right) / 2.0

        return {
            "path": path,
            "class": os.path.basename(os.path.dirname(path)),
            "smile_left": smile_left,
            "smile_right": smile_right,
            "smile_score": smile_score,
        }

    except Exception:
        return None


# Считаем score для всех кандидатов
rows = []
for path in face_like_paths:
    row = get_smile_score(path)
    if row is not None:
        rows.append(row)

smile_df = pd.DataFrame(rows).sort_values("smile_score", ascending=False).reset_index(drop=True)

print("Всего лиц обработано:", len(smile_df))
display(smile_df.head(10))

In [ ]:
# визуализируем эти лица
fig, axs = plt.subplots(4, 5, figsize=(15, 12))
for ax, (_, row) in zip(axs.ravel(), smile_df.head(20).iterrows()):
    ax.set_title(f"{row['class']}\nsmile_score: {row['smile_score']:.3f}")
    ax.imshow(plt.imread(row["path"]))
    ax.axis('off')
plt.tight_layout()
plt.show()

### 4.2 Поиск похожих лиц

Для поиска похожих лиц также будем использовать поиск по эмбеддингам, но для генерации эмбеддингов будем использовать специализированную модель ArcFace (InsightFace).

In [ ]:
%pip install insightface onnxruntime

In [ ]:
from sklearn.preprocessing import normalize

import insightface
from insightface.app import FaceAnalysis

In [ ]:
face_app = FaceAnalysis(
    name="buffalo_s",
    # providers=["CPUExecutionProvider"],
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"],
)
face_app.prepare(ctx_id=0, det_size=(320, 320))


def load_rgb(path: str) -> np.ndarray:
    return np.array(Image.open(path).convert("RGB"))


def get_face_embeddings(paths, min_det_score=0.50) -> pd.DataFrame:
    """
    Возвращает DataFrame:
      index = path
      columns = face_0 ... face_N
    Берем только самое уверенное лицо на изображении.
    """
    rows = []

    for path in paths:
        try:
            img = load_rgb(path)
            faces = face_app.get(img)

            if len(faces) == 0:
                continue

            # берем лицо с максимальным score / det_score
            best_face = None
            best_score = -1.0

            for f in faces:
                score = getattr(f, "det_score", 0.0)
                if score > best_score:
                    best_score = score
                    best_face = f

            if best_face is None or best_score < min_det_score:
                continue

            emb = np.asarray(best_face.embedding, dtype=np.float32)
            rows.append((path, emb))

        except Exception:
            # пропускаем проблемные изображения
            continue

    if not rows:
        return pd.DataFrame()

    index = [r[0] for r in rows]
    X = np.vstack([r[1] for r in rows])

    # Нормализация обязательна для cosine similarity
    X = normalize(X, norm="l2")

    df_faces = pd.DataFrame(
        X,
        index=index,
        columns=[f"face_{i}" for i in range(X.shape[1])]
    )
    return df_faces


# Для Caltech101 имеет смысл сначала попробовать только "лицевые" категории
face_like_paths = [
    p for p in df.index
    if os.path.basename(os.path.dirname(p)) in {"Faces", "Faces_easy"}
]

df_face = get_face_embeddings(face_like_paths)
df_face

In [ ]:
# del face_app

In [ ]:
neighbors_face = NearestNeighbors(
    n_neighbors=min(8, len(df_face)),
    metric="cosine",
    algorithm="brute"
).fit(df_face.values)


def show_face_neighbors(query_path: str, df_face: pd.DataFrame, knn, n_show=8):
    query_vec = df_face.loc[[query_path]].values
    distances, indices = knn.kneighbors(query_vec, n_neighbors=min(n_show, len(df_face)))

    found_paths = df_face.iloc[indices[0]].index.tolist()
    found_dist = distances[0]

    fig = plt.figure(figsize=(3 * len(found_paths), 4))

    for i, (path, dist) in enumerate(zip(found_paths, found_dist), start=1):
        ax = fig.add_subplot(1, len(found_paths), i)
        ax.imshow(Image.open(path).convert("RGB"))
        ax.axis("off")
        cls = os.path.basename(os.path.dirname(path))
        ax.set_title(f"{cls}\ncos dist={dist:.3f}")

    plt.tight_layout()
    plt.show()


# np.random.seed(42)
# query_face_path = np.random.choice(df_face.index.to_numpy())
query_face_path = "./datasets/caltech-101/Faces/image_0076.jpg"

print("Запрос:", query_face_path)
show_face_neighbors(query_face_path, df_face, neighbors_face, n_show=8)

**ПРАКТИКА**

Загрузите собственную фотографию и найдите похожие лица в датасете CalTech101. Выведите на экран найденные лица и сделайте выводы.

In [ ]:
# ваш код здесь



